<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day4/Session4/session4_compare_and_share.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4 — Did It Work? Evidence & Sharing

**Day 4 · 15:30 – 16:30**

Compare honestly, publish your adapter, and learn when fine-tuning is the wrong tool.

> **Continue in the same Colab session as Session 3 if you can** — you already have the
> trained model in memory. If you restarted, run Cell 1B to reload from your saved adapter.

## Cell 1A — If you are continuing from Session 3

Skip to Cell 2. Your `model`, `before` and `QUESTIONS` are already loaded.

## Cell 1B — If Colab restarted, reload everything

You need the `qwen-amharic-lora` folder. If you downloaded it, upload it back using the
file panel on the left.

In [ ]:
!pip install -q transformers datasets accelerate peft

import torch, pickle
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL = "Qwen/Qwen2-0.5B-Instruct"
ADAPTER = "./qwen-amharic-lora"

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto")

model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()

with open("baseline.pkl", "rb") as f:
    saved = pickle.load(f)
QUESTIONS = saved["questions"]
before    = saved["before"]

print("Reloaded model and baseline")

## Cell 2 — The helper we use to ask questions

In [ ]:
def ask(m, question, max_new_tokens=120):
    messages = [
        {"role": "system", "content": "You are a helpful Ethiopian customer service assistant."},
        {"role": "user",   "content": question},
    ]
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs,
                         max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("ready")

## Cell 3 — BEFORE vs AFTER

The moment of truth. Same questions, same model, one thing changed.

In [ ]:
model.eval()

after = {}
for q in QUESTIONS:
    after[q] = ask(model, q)

for i, q in enumerate(QUESTIONS, 1):
    print("=" * 70)
    print(f"QUESTION {i}: {q}")
    print("=" * 70)
    print("BEFORE:")
    print("  ", before[q][:250].replace("\n", "\n   "))
    print()
    print("AFTER:")
    print("  ", after[q][:250].replace("\n", "\n   "))
    print()

## Cell 4 — Judge it honestly

Score each answer yourself on three things. Be strict.

| Question | Before | After |
|---|---|---|
| Did it answer **in Amharic**? | | |
| Did it **stop**, or ramble? | | |
| Does it sound like **customer service**? | | |

### Set your expectations correctly

Ten minutes of training on 1,000 examples produces a **real but modest** change.
What usually improves:

- ✅ Answering in Amharic more consistently
- ✅ Following the question/answer format
- ✅ Stopping instead of rambling

What usually does **not** improve:

- ❌ Factual knowledge it never had
- ❌ Reasoning ability
- ❌ Fluency beyond what the base model could already do

**If you claim more than you can show, that is marketing, not engineering.**
Being able to say clearly what did *not* improve is a professional skill.

## Cell 5 — Measure something objective

Judgement is subjective. Here is one number you can actually count:
how much of the reply is in Ge'ez script?

In [ ]:
def amharic_ratio(text):
    if not text.strip():
        return 0.0
    # Ethiopic Unicode block is U+1200 to U+137F
    ethiopic = sum(1 for ch in text if "\u1200" <= ch <= "\u137f")
    letters  = sum(1 for ch in text if ch.isalpha() or "\u1200" <= ch <= "\u137f")
    return ethiopic / letters if letters else 0.0


print(f"{'Question':<12}{'BEFORE':>10}{'AFTER':>10}")
print("-" * 34)
b_tot = a_tot = 0
for i, q in enumerate(QUESTIONS, 1):
    b = amharic_ratio(before[q]); a = amharic_ratio(after[q])
    b_tot += b; a_tot += a
    print(f"Q{i:<11}{b:>9.0%}{a:>10.0%}")
print("-" * 34)
n = len(QUESTIONS)
print(f"{'AVERAGE':<12}{b_tot/n:>9.0%}{a_tot/n:>10.0%}")

> **Careful with this number.** A higher Amharic ratio means the model *replied in
> Amharic more often* — it does **not** mean the answers are correct or useful.
> Measuring the easy thing and claiming it proves the hard thing is a classic mistake.

## Cell 6 — Publish your adapter to Hugging Face

Your 5 MB adapter can be used by anyone in the world, in three lines of code.

**First get a token:**
1. Go to <https://huggingface.co/settings/tokens>
2. **New token** → role: **WRITE** → create
3. Copy it (shown only once)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()   # paste your WRITE token in the box that appears

In [ ]:
# Change to YOUR Hugging Face username
HF_USERNAME = "your-username"
REPO_NAME   = "qwen2-0.5b-amharic-lora"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tok.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"Published: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

### Anyone can now use your work

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
model = PeftModel.from_pretrained(base, "YOUR-USERNAME/qwen2-0.5b-amharic-lora")
```

That is a real contribution to Amharic AI. Someone in Bahir Dar can build on it tomorrow.

## Cell 7 — When NOT to fine-tune

The most valuable thing you can learn today. **Try these in order** — most problems
stop at step 1.

| Approach | Cost | Time | Use it when |
|---|---|---|---|
| **1. Better prompt** | Free | Minutes | Almost always try first |
| **2. RAG** (look facts up) | Cheap | Hours | The model needs *facts* it does not have |
| **3. Fine-tuning** | Real GPU cost | Days | You need consistent *style, format or language* |

### Fine-tuning is good at
- Style, tone and output format
- Speaking a language more consistently
- Following your organisation's specific instructions

### Fine-tuning is bad at
- Adding facts (use RAG — it looks them up instead)
- Fixing what a better prompt would fix
- Making a small model as clever as a large one
- Compensating for bad or tiny data

**A common expensive mistake:** fine-tuning a model to memorise a company handbook.
RAG does that better, cheaper, and updates instantly when the handbook changes.

## Cell 8 — Clean up and check what you are taking home

In [ ]:
import gc

print("Files in this session:")
!ls -la

print()
print("Adapter size:")
!du -sh ./qwen-amharic-lora 2>/dev/null || echo "  (not found - did you save it?)"

gc.collect()
torch.cuda.empty_cache()
print()
print("REMINDER: Colab deletes everything when this session ends.")
print("Download ./qwen-amharic-lora from the file panel, or push it to Hugging Face.")

---

## Day 4 — what you actually did

```
Measured the GPU limit          ->  15 GB is the wall
Calculated before downloading   ->  arithmetic beats guessing
Crashed a model on purpose      ->  OOM is normal, not fatal
Measured the Amharic penalty    ->  3x more tokens than English
Recorded a baseline             ->  evidence, not claims
Attached LoRA adapters          ->  0.2% trainable
Trained on a free GPU           ->  ~10 minutes
Compared honestly               ->  said what did NOT improve
Published to Hugging Face       ->  contributed to Amharic AI
```

## Three ideas to remember

1. **Constraints are the skill.** Anyone can train with unlimited GPUs. Making it work
   in 15 GB is the job you will be paid for.
2. **Prove it, do not claim it.** Baseline first, compare honestly, state the limits.
3. **Small, local and yours.** A 0.5B model fine-tuned on Ethiopian data, running on
   Ethiopian hardware, answering in Amharic — that is the whole point.

**Tomorrow — Day 5:** deployment and the capstone.